In [1]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

churn_data = pd.read_csv('telco_churn_data.csv')

In [2]:
# Separate features and target variable
X = churn_data.drop('Customer Status', axis=1)
y = churn_data['Customer Status']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [3]:
# Define models 🤖
models = {
    'Logistic Regression': OneVsRestClassifier(LogisticRegression(random_state=42)),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    # 'SVM': OneVsRestClassifier(SVC(probability=True, random_state=42)),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42)
}

# Define balancing techniques ⚖️
balancing_methods = {
    'None': None,
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'Random Over Sampling': RandomOverSampler(random_state=42),
    'Random Under Sampling': RandomUnderSampler(random_state=42),
    'Tomek Links': TomekLinks(),
    'SMOTE-Tomek': SMOTETomek(random_state=42)
}

# Results dictionary 📊
results = {}

In [4]:
# Iterate over balancing methods and models
for balancer_name, balancer in balancing_methods.items():
    results[balancer_name] = {}
    # Apply balancing to training data
    if balancer:
        X_train_balanced, y_train_balanced = balancer.fit_resample(X_train, y_train)
    else:
        X_train_balanced, y_train_balanced = X_train, y_train

    for model_name, model in models.items():
        # Train and evaluate model
        model.fit(X_train_balanced, y_train_balanced)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test) #get probabilities

        # Store metrics
        results[balancer_name][model_name] = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'f1_score': f1_score(y_test, y_pred, average='weighted'),
            'roc_auc_score': roc_auc_score(y_test, y_pred_proba, multi_class='ovr'),
            'confusion_matrix': confusion_matrix(y_test, y_pred)
        }